# LeetCode 1712 — Ways to Split Array Into Three Subarrays bằng Python

Link đề: https://leetcode.com/problems/ways-to-split-array-into-three-subarrays/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def waysToSplit(self, nums: List[int]) -> int:
        ...
```

Lộ trình: vét cạn `O(n³)` → tổng tiền tố `O(n²)` → **tổng tiền tố + tìm nhị phân** `O(n log n)` → **ba con trỏ** `O(n)`. Chìa khóa: vì mọi phần tử **không âm**, tổng tiền tố **không giảm**, nên tập các vị trí cắt thứ hai hợp lệ là **một đoạn liên tiếp**.

## 1. Mục tiêu học

1. Biểu diễn "chia mảng thành 3 đoạn liên tiếp" bằng **hai vị trí cắt** `i < j` và viết tổng mỗi đoạn qua **mảng tổng tiền tố** `P`.
2. Biến hai bất đẳng thức `left ≤ mid ≤ right` thành **chặn dưới** và **chặn trên** cho `P[j]`: `2·P[i] ≤ P[j] ≤ (T + P[i]) / 2`.
3. Dùng tính **đơn điệu** của `P` (do `nums[k] ≥ 0`) để tìm cả đoạn `j` hợp lệ bằng `bisect_left` / `bisect_right`.
4. Nâng cấp thành **hai con trỏ chỉ tiến** (thêm con trỏ `i` là ba) đạt `O(n)`, và hiểu vì sao các con trỏ không bao giờ phải lùi.
5. Cẩn thận các biên: mỗi đoạn **không rỗng**, dấu `≤` (không phải `<`), số đếm không âm, và `mod 10⁹ + 7`.

## 2. Đề bài & ví dụ

Một cách chia mảng số nguyên là **tốt** nếu:
- Mảng được chia thành 3 mảng con **liên tiếp, không rỗng**, lần lượt là `left`, `mid`, `right` (từ trái sang phải).
- `sum(left) ≤ sum(mid)` **và** `sum(mid) ≤ sum(right)`.

Cho `nums` gồm các số nguyên **không âm**, trả về **số cách chia tốt**, lấy modulo `10⁹ + 7`.

**Ví dụ 1:** `nums = [1,1,1]` → `1`. Cách duy nhất: `[1] [1] [1]`.

**Ví dụ 2:** `nums = [1,2,2,2,5,0]` → `3`:
- `[1] [2] [2,2,5,0]`
- `[1] [2,2] [2,5,0]`
- `[1,2] [2,2] [5,0]`

**Ví dụ 3:** `nums = [3,2,1]` → `0`.

**Ràng buộc:**
- `3 ≤ nums.length ≤ 10⁵`
- `0 ≤ nums[i] ≤ 10⁴`

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from itertools import accumulate
from typing import List

MOD = 10**9 + 7

examples = [
    ([1, 1, 1], 1),
    ([1, 2, 2, 2, 5, 0], 3),
    ([3, 2, 1], 0),
]
for nums, want in examples:
    print(nums, "-> kỳ vọng", want)

[1, 1, 1] -> kỳ vọng 1
[1, 2, 2, 2, 5, 0] -> kỳ vọng 3
[3, 2, 1] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Hai vị trí cắt

Với mảng độ dài `n`, chọn `1 ≤ i < j ≤ n − 1`:
- `left = nums[0 : i]` (có `i ≥ 1` phần tử),
- `mid = nums[i : j]` (có `j − i ≥ 1` phần tử),
- `right = nums[j : n]` (có `n − j ≥ 1` phần tử).

Tổng số cách chọn `(i, j)` là `C(n − 1, 2)` — đây cũng là chặn trên của đáp án.

In [2]:
nums = [1, 2, 2, 2, 5, 0]
n = len(nums)
for i in range(1, n - 1):
    for j in range(i + 1, n):
        print((i, j), nums[:i], nums[i:j], nums[j:])
    if i == 2:
        print("...")
        break

(1, 2) [1] [2] [2, 2, 5, 0]
(1, 3) [1] [2, 2] [2, 5, 0]
(1, 4) [1] [2, 2, 2] [5, 0]
(1, 5) [1] [2, 2, 2, 5] [0]
(2, 3) [1, 2] [2] [2, 5, 0]
(2, 4) [1, 2] [2, 2] [5, 0]
(2, 5) [1, 2] [2, 2, 5] [0]
...


### 3.2 Tổng tiền tố `P`

Định nghĩa `P[0] = 0`, `P[k] = nums[0] + … + nums[k − 1]`. Khi đó tổng đoạn `nums[a : b]` = `P[b] − P[a]` trong `O(1)`. Trong Python: `P = [0] + list(accumulate(nums))`.

Với hai vị trí cắt `i, j` và `T = P[n]`:
- `sum(left) = P[i]`
- `sum(mid) = P[j] − P[i]`
- `sum(right) = T − P[j]`

In [3]:
nums = [1, 2, 2, 2, 5, 0]
P = [0] + list(accumulate(nums))
T = P[-1]
print("P =", P, "| T =", T)
i, j = 2, 4
print("left =", P[i], "| mid =", P[j] - P[i], "| right =", T - P[j])
print("kiểm tra bằng sum:", sum(nums[:i]), sum(nums[i:j]), sum(nums[j:]))

P = [0, 1, 3, 5, 7, 12, 12] | T = 12
left = 3 | mid = 4 | right = 5
kiểm tra bằng sum: 3 4 5


### 3.3 Mảng không âm ⇒ `P` không giảm

Vì `nums[k] ≥ 0`, `P[k + 1] = P[k] + nums[k] ≥ P[k]`. Một mảng **không giảm** (có thể có giá trị bằng nhau liên tiếp khi có số 0) cho phép **tìm nhị phân**.

### 3.4 `bisect_left` và `bisect_right` với khoảng `[lo, hi)`

- `bisect_left(a, x, lo, hi)`: chỉ số nhỏ nhất `k ∈ [lo, hi]` mà `a[k] ≥ x` (hoặc `hi` nếu không có).
- `bisect_right(a, x, lo, hi)`: chỉ số nhỏ nhất `k ∈ [lo, hi]` mà `a[k] > x`.

Vậy các chỉ số `k ∈ [lo, hi)` có `L ≤ a[k] ≤ R` là đoạn `[bisect_left(a, L, lo, hi), bisect_right(a, R, lo, hi))`.

In [4]:
a = [0, 1, 3, 5, 7, 12, 12]
print("bisect_left(a, 6)  =", bisect_left(a, 6))
print("bisect_right(a, 7) =", bisect_right(a, 7))
print("chỉ số k thuộc [2, 6) với 2 <= a[k] <= 6:",
      list(range(bisect_left(a, 2, 2, 6), bisect_right(a, 6, 2, 6))))

bisect_left(a, 6)  = 4
bisect_right(a, 7) = 5
chỉ số k thuộc [2, 6) với 2 <= a[k] <= 6: [2, 3]


### 3.5 Chia nguyên và bất đẳng thức

Với số nguyên, `2·x ≤ S` ⇔ `x ≤ S // 2` (chia lấy phần nguyên xuống, khi `S ≥ 0`). Ta dùng điều này để đưa chặn trên `P[j] ≤ (T + P[i]) / 2` về số nguyên `(T + P[i]) // 2`, tránh dùng số thực.

In [5]:
for S in range(0, 8):
    xs_ok = [x for x in range(0, 6) if 2 * x <= S]
    print(f"S={S}: x với 2x <= S là {xs_ok}, S//2 = {S // 2}")

S=0: x với 2x <= S là [0], S//2 = 0
S=1: x với 2x <= S là [0], S//2 = 0
S=2: x với 2x <= S là [0, 1], S//2 = 1
S=3: x với 2x <= S là [0, 1], S//2 = 1
S=4: x với 2x <= S là [0, 1, 2], S//2 = 2
S=5: x với 2x <= S là [0, 1, 2], S//2 = 2
S=6: x với 2x <= S là [0, 1, 2, 3], S//2 = 3
S=7: x với 2x <= S là [0, 1, 2, 3], S//2 = 3


## 4. Ý tưởng nền tảng / trực giác

**Tính tay Ví dụ 2** `[1,2,2,2,5,0]`, `P = [0,1,3,5,7,12,12]`, `T = 12`:

| `i` | `left = P[i]` | cần `P[j] ≥ 2·P[i]` | cần `P[j] ≤ (T+P[i])//2` | `j` hợp lệ (trong `i+1..5`) | số cách |
|---|---|---|---|---|---|
| 1 | 1 | `P[j] ≥ 2` | `P[j] ≤ 6` | `j = 2 (P=3), 3 (P=5)` | 2 |
| 2 | 3 | `P[j] ≥ 6` | `P[j] ≤ 7` | `j = 4 (P=7)` | 1 |
| 3 | 5 | `P[j] ≥ 10` | `P[j] ≤ 8` | không có | 0 |
| 4 | 7 | `P[j] ≥ 14` | `P[j] ≤ 9` | không có | 0 |

Tổng `3` ✔.

**Hai quan sát:**
1. Với `i` cố định, các `j` hợp lệ tạo **một đoạn liên tiếp** (vì `P` không giảm, và điều kiện là "`P[j]` nằm giữa hai số").
2. Khi `i` tăng, **cả hai đầu** của đoạn chỉ có thể **dịch sang phải** (vì `2·P[i]` và `(T + P[i]) // 2` đều không giảm theo `i`).

Quan sát 1 ⇒ tìm nhị phân. Quan sát 2 ⇒ hai con trỏ.

In [6]:
nums = [1, 2, 2, 2, 5, 0]
n = len(nums)
P = [0] + list(accumulate(nums)); T = P[-1]
for i in range(1, n - 1):
    ok = [j for j in range(i + 1, n) if 2 * P[i] <= P[j] <= (T + P[i]) // 2]
    print(f"i={i}: P[i]={P[i]}, đoạn [{2 * P[i]}, {(T + P[i]) // 2}] -> j hợp lệ {ok}")

i=1: P[i]=1, đoạn [2, 6] -> j hợp lệ [2, 3]
i=2: P[i]=3, đoạn [6, 7] -> j hợp lệ [4]
i=3: P[i]=5, đoạn [10, 8] -> j hợp lệ []
i=4: P[i]=7, đoạn [14, 9] -> j hợp lệ []


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp cắt, tính tổng bằng `sum`

**Ý tưởng:** duyệt mọi `(i, j)`, tính ba tổng bằng `sum` của lát cắt, đếm cặp thỏa.

In [7]:
def ways_brute(nums):
    n = len(nums)
    ans = 0
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            l, m, r = sum(nums[:i]), sum(nums[i:j]), sum(nums[j:])
            if l <= m <= r:
                ans += 1
    return ans % MOD

for nums, want in examples:
    print(ways_brute(nums), "| kỳ vọng", want)

1 | kỳ vọng 1
3 | kỳ vọng 3
0 | kỳ vọng 0


**Độ phức tạp:** `O(n²)` cặp × `O(n)` cho `sum` ⇒ Time `O(n³)`, Space `O(n)` (lát cắt tạo bản sao).

**Nút thắt:** tính lại tổng từ đầu cho mỗi cặp. Với `n = 10⁵` thì `10¹⁵` phép — không thể.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — bỏ `sum` lặp lại:** dùng `P`. Điều kiện thành:
- `left ≤ mid` ⇔ `P[i] ≤ P[j] − P[i]` ⇔ **`P[j] ≥ 2·P[i]`**
- `mid ≤ right` ⇔ `P[j] − P[i] ≤ T − P[j]` ⇔ `2·P[j] ≤ T + P[i]` ⇔ **`P[j] ≤ (T + P[i]) // 2`**
- và ràng buộc vị trí: **`i + 1 ≤ j ≤ n − 1`**.

**Bước 2 — đoạn liên tiếp:** vì `P` không giảm, `{ j : lo_val ≤ P[j] ≤ hi_val }` là một đoạn chỉ số liên tiếp, đếm được bằng **2 lần tìm nhị phân**.

**Bước 3 — dừng sớm:** `left ≤ mid ≤ right` ⇒ `3·left ≤ T` ⇒ nếu `3·P[i] > T` thì mọi `i` lớn hơn cũng hỏng (vì `P` không giảm) ⇒ `break`.

**Bước 4 — con trỏ đơn điệu:** khi `i` tăng, `2·P[i]` và `(T + P[i]) // 2` không giảm, chặn `i + 1` cũng tăng. Vậy chỉ số đầu đoạn `lo(i)` và chỉ số sau-cuối-đoạn `hi(i)` đều **không giảm** theo `i` ⇒ mỗi con trỏ đi tối đa `n` bước tổng cộng.

> **Quy tắc:** *Nếu* `P[j] < 2·P[i]` *thì* tăng `lo`; *nếu* `2·P[hi] ≤ T + P[i]` *thì* tăng `hi`; số cách với `i` là `max(0, hi − lo)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tổng tiền tố, vẫn thử mọi cặp `O(n²)`

Chỉ thay `sum` bằng phép trừ trên `P`. Đây là "brute force thông minh" dùng làm **đáp án chuẩn** cho test với `n` cỡ vài trăm.

In [8]:
def ways_prefix_n2(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    T = P[n]
    ans = 0
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            if 2 * P[i] <= P[j] and 2 * P[j] <= T + P[i]:
                ans += 1
    return ans % MOD

for nums, want in examples:
    print(ways_prefix_n2(nums), "| kỳ vọng", want)

1 | kỳ vọng 1
3 | kỳ vọng 3
0 | kỳ vọng 0


**Độ phức tạp Cách 2:** Time `O(n²)`, Space `O(n)`. Vẫn TLE với `n = 10⁵` (`5·10⁹` cặp).

### 7.2 Cách 3 — Tổng tiền tố + tìm nhị phân `O(n log n)`

**Từng đoạn:**
1. Tính `P`, `T`.
2. Với mỗi `i = 1 … n − 2`:
   - nếu `3·P[i] > T`: `break`;
   - `lo = bisect_left(P, 2·P[i], i + 1, n)` — `j` nhỏ nhất (≥ `i+1`) có `P[j] ≥ 2·P[i]`;
   - `hi = bisect_right(P, (T + P[i]) // 2, lo, n)` — `j` nhỏ nhất (≥ `lo`) có `P[j]` vượt chặn trên;
   - cộng `hi − lo` (luôn `≥ 0` vì tìm `hi` bắt đầu từ `lo`).
3. Trả `ans % MOD`.

Chú ý tham số `hi = n` của `bisect` nghĩa là chỉ xét chỉ số `≤ n − 1` — đúng điều kiện `right` không rỗng.

In [9]:
def ways_bisect(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    T = P[n]
    ans = 0
    for i in range(1, n - 1):
        if 3 * P[i] > T:
            break
        lo = bisect_left(P, 2 * P[i], i + 1, n)
        hi = bisect_right(P, (T + P[i]) // 2, lo, n)
        ans += hi - lo
    return ans % MOD

for nums, want in examples:
    print(ways_bisect(nums), "| kỳ vọng", want)

1 | kỳ vọng 1
3 | kỳ vọng 3
0 | kỳ vọng 0


Code nộp LeetCode của Cách 3:

In [10]:
class SolutionBisect:
    def waysToSplit(self, nums: List[int]) -> int:
        n = len(nums)
        P = [0] + list(accumulate(nums))
        T = P[n]
        ans = 0
        for i in range(1, n - 1):
            if 3 * P[i] > T:
                break
            lo = bisect_left(P, 2 * P[i], i + 1, n)
            hi = bisect_right(P, (T + P[i]) // 2, lo, n)
            ans += hi - lo
        return ans % (10**9 + 7)

print([SolutionBisect().waysToSplit(x) for x, _ in examples])

[1, 3, 0]


**Độ phức tạp Cách 3:** Time `O(n log n)`, Space `O(n)`.

### 7.3 Cách 4 — Ba con trỏ `O(n)`

**Từng đoạn:**
1. `lo = hi = 0` là hai con trỏ cho `j`; `i` là con trỏ thứ ba (vòng for).
2. Với mỗi `i`:
   - `lo = max(lo, i + 1)` (đoạn `mid` không rỗng);
   - tiến `lo` khi `lo ≤ n − 1` và `P[lo] < 2·P[i]`;
   - `hi = max(hi, lo)`; tiến `hi` khi `hi ≤ n − 1` và `2·P[hi] ≤ T + P[i]`;
   - lúc này `[lo, hi)` là đoạn `j` hợp lệ ⇒ `ans += hi − lo`.
3. Vì `lo`, `hi` chỉ tăng và bị chặn bởi `n`, tổng số bước là `O(n)`.

Lý do cần `hi = max(hi, lo)`: nếu đoạn hợp lệ của `i` trước đó rỗng, `hi` có thể **đứng sau** `lo`; khi đó `hi − lo` âm. Kéo `hi` lên `lo` không làm mất `j` hợp lệ nào (mọi `j < lo` đều đã bị loại bởi chặn dưới).

In [11]:
def ways_three_pointers(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    T = P[n]
    ans = 0
    lo = hi = 0
    for i in range(1, n - 1):
        if 3 * P[i] > T:
            break
        lo = max(lo, i + 1)
        while lo <= n - 1 and P[lo] < 2 * P[i]:
            lo += 1
        hi = max(hi, lo)
        while hi <= n - 1 and 2 * P[hi] <= T + P[i]:
            hi += 1
        ans += hi - lo
    return ans % MOD

for nums, want in examples:
    print(ways_three_pointers(nums), "| kỳ vọng", want)

1 | kỳ vọng 1
3 | kỳ vọng 3
0 | kỳ vọng 0


Code nộp LeetCode của Cách 4 (bản nộp chính, xem lại ở mục 9).

**Độ phức tạp Cách 4:** Time `O(n)` (mỗi con trỏ tiến tối đa `n` lần), Space `O(n)` cho `P` (có thể `O(1)` nếu tính tổng chạy, nhưng code kém rõ ràng).

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 2: `P = [0,1,3,5,7,12,12]`, `T = 12`, `n = 6` (nên `j ∈ [i+1, 5]`).

| `i` | `P[i]` | chặn dưới `2P[i]` | chặn trên `(T+P[i])//2` | `lo` | `hi` | `hi − lo` | `ans` |
|---|---|---|---|---|---|---|---|
| 1 | 1 | 2 | 6 | 2 | 4 | 2 | 2 |
| 2 | 3 | 6 | 7 | 4 | 5 | 1 | 3 |
| 3 | 5 | 10 | 8 | 5 | 5 | 0 | 3 |
| 4 | 7 | — | — | `3·7 = 21 > 12` ⇒ dừng | | | 3 |

In [12]:
def trace_three_pointers(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    T = P[n]
    print("P =", P, "T =", T)
    ans = 0
    lo = hi = 0
    for i in range(1, n - 1):
        if 3 * P[i] > T:
            print(f"i={i}: 3*P[i]={3 * P[i]} > T={T} -> dừng")
            break
        lo = max(lo, i + 1)
        while lo <= n - 1 and P[lo] < 2 * P[i]:
            lo += 1
        hi = max(hi, lo)
        while hi <= n - 1 and 2 * P[hi] <= T + P[i]:
            hi += 1
        ans += hi - lo
        js = list(range(lo, hi))
        print(f"i={i}: P[i]={P[i]:>2} cần {2 * P[i]} <= P[j] <= {(T + P[i]) // 2}  lo={lo} hi={hi} "
              f"j={js} +{hi - lo} ans={ans}")
        for j in js:
            print("      ", nums[:i], nums[i:j], nums[j:])
    return ans

print("Kết quả:", trace_three_pointers([1, 2, 2, 2, 5, 0]))

P = [0, 1, 3, 5, 7, 12, 12] T = 12
i=1: P[i]= 1 cần 2 <= P[j] <= 6  lo=2 hi=4 j=[2, 3] +2 ans=2
       [1] [2] [2, 2, 5, 0]
       [1] [2, 2] [2, 5, 0]
i=2: P[i]= 3 cần 6 <= P[j] <= 7  lo=4 hi=5 j=[4] +1 ans=3
       [1, 2] [2, 2] [5, 0]
i=3: 3*P[i]=15 > T=12 -> dừng
Kết quả: 3


Trace trên mảng có nhiều số `0` — nơi `P` có các đoạn bằng nhau và đáp án lớn:

In [13]:
print("Kết quả:", trace_three_pointers([0, 0, 0, 0, 0]))

P = [0, 0, 0, 0, 0, 0] T = 0
i=1: P[i]= 0 cần 0 <= P[j] <= 0  lo=2 hi=5 j=[2, 3, 4] +3 ans=3
       [0] [0] [0, 0, 0]
       [0] [0, 0] [0, 0]
       [0] [0, 0, 0] [0]
i=2: P[i]= 0 cần 0 <= P[j] <= 0  lo=3 hi=5 j=[3, 4] +2 ans=5
       [0, 0] [0] [0, 0]
       [0, 0] [0, 0] [0]
i=3: P[i]= 0 cần 0 <= P[j] <= 0  lo=4 hi=5 j=[4] +1 ans=6
       [0, 0, 0] [0] [0]
Kết quả: 6


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def waysToSplit(self, nums: List[int]) -> int:
        MOD = 10**9 + 7
        n = len(nums)
        P = [0] * (n + 1)
        for k, x in enumerate(nums):
            P[k + 1] = P[k] + x
        T = P[n]
        ans = 0
        lo = hi = 0
        for i in range(1, n - 1):
            if 3 * P[i] > T:                    # left đã quá 1/3 tổng
                break
            lo = max(lo, i + 1)                 # mid không rỗng
            while lo <= n - 1 and P[lo] < 2 * P[i]:
                lo += 1                         # chặn dưới: left <= mid
            hi = max(hi, lo)
            while hi <= n - 1 and 2 * P[hi] <= T + P[i]:
                hi += 1                         # chặn trên: mid <= right
            ans += hi - lo
        return ans % MOD

sol = Solution()
for nums, want in examples:
    assert sol.waysToSplit(nums) == want
print("Solution qua 3 ví dụ của đề")

Solution qua 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

**Tương đương điều kiện.** Với `1 ≤ i < j ≤ n − 1`, cách chia `(i, j)` tốt ⇔ `2·P[i] ≤ P[j]` và `2·P[j] ≤ T + P[i]` (biến đổi đại số ở mục 6, không chia số thực nên không mất chính xác).

**Đoạn liên tiếp.** Cố định `i`. Đặt `A(i) = { j ∈ [i+1, n−1] : P[j] ≥ 2·P[i] }`, `B(i) = { j ∈ [i+1, n−1] : 2·P[j] ≤ T + P[i] }`. Vì `P` không giảm: `A(i)` là một **hậu tố** của `[i+1, n−1]`, `B(i)` là một **tiền tố**. Giao của tiền tố và hậu tố là một đoạn `[lo(i), hi(i))` (có thể rỗng). Số `j` hợp lệ = `max(0, hi(i) − lo(i))`.

**Con trỏ đơn điệu (Cách 4).** Khi `i` tăng lên `i + 1`:
- chặn dưới `max(i + 1, ·)` và `2·P[i]` không giảm ⇒ `lo(i + 1) ≥ lo(i)`;
- chặn trên `(T + P[i]) // 2` không giảm ⇒ tiền tố `B` chỉ dài ra ⇒ điểm kết thúc `hi` không giảm.

Vì vậy tiếp tục con trỏ từ vị trí cũ (không reset) vẫn tìm đúng `lo(i)`, `hi(i)`. Thao tác `hi = max(hi, lo)` chỉ bỏ qua các `j < lo(i)` — vốn không hợp lệ.

**Dừng sớm.** Nếu `3·P[i] > T` thì với mọi `j`: `left ≤ mid ≤ right` sẽ kéo theo `T = left + mid + right ≥ 3·left > T`, mâu thuẫn; và `P[i']` với `i' > i` còn lớn hơn nữa.

## 11. Phân tích Time / Space Complexity

Gọi `n` = độ dài mảng (`3 ≤ n ≤ 10⁵`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn `sum` | `O(n³)` | `O(n)` | chỉ dùng `n ≤ 60` |
| 2. Tiền tố, mọi cặp | `O(n²)` | `O(n)` | đáp án chuẩn cho test vừa |
| 3. Tiền tố + bisect | `O(n log n)` | `O(n)` | an toàn, dễ viết đúng |
| 4. Ba con trỏ | `O(n)` | `O(n)` | **tối ưu** |

Đáp án có thể tới `C(n − 1, 2) ≈ 5·10⁹` (mảng toàn `0`) ⇒ phải lấy modulo. Không có worst/average khác biệt: các cách đều tất định.

In [15]:
rng = random.Random(1712)
big = [rng.randint(0, 10**4) for _ in range(10**5)]
zeros = [0] * 10**5
for name, f in [("bisect", ways_bisect), ("3 con trỏ", ways_three_pointers), ("Solution", Solution().waysToSplit)]:
    for label, arr in [("ngẫu nhiên", big), ("toàn 0", zeros)]:
        t = time.perf_counter()
        r = f(arr)
        print(f"{name:>10} | {label:<10}: {time.perf_counter() - t:.3f}s ans={r}")

    bisect | ngẫu nhiên: 0.015s ans=832886079
    bisect | toàn 0    : 0.028s ans=999849973
 3 con trỏ | ngẫu nhiên: 0.019s ans=832886079
 3 con trỏ | toàn 0    : 0.020s ans=999849973
  Solution | ngẫu nhiên: 0.019s ans=832886079


  Solution | toàn 0    : 0.023s ans=999849973


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

- `[0,0,0]` ⇒ 1 (chỉ một cách chia, `0 ≤ 0 ≤ 0`).
- `[0,0,0,0]` ⇒ `C(3,2) = 3`.
- `[1,0,0,1]`: thử tay — `[1][0][0,1]` có `1 ≤ 0`? Không. … ⇒ 0.
- `[0,0,1]` ⇒ 1; `[1,0,0]` ⇒ 0.
- `[2,2,2,2]` ⇒ 1 (chỉ `[2][2][2,2]`).
- `[0]*5` ⇒ `C(4,2) = 6`.

In [16]:
impls = {
    "brute": ways_brute,
    "prefix_n2": ways_prefix_n2,
    "bisect": ways_bisect,
    "three": ways_three_pointers,
    "Solution": Solution().waysToSplit,
    "SolutionBisect": SolutionBisect().waysToSplit,
}
edge = [
    ([0, 0, 0], 1), ([0, 0, 0, 0], 3), ([1, 0, 0, 1], 0), ([0, 0, 1], 1),
    ([1, 0, 0], 0), ([2, 2, 2, 2], 1), ([0] * 5, 6), ([10**4] * 3, 1),
]
for nums, want in examples + edge:
    for name, f in impls.items():
        assert f(nums) == want, (name, nums, f(nums), want)
print("Tất cả cách qua", len(examples) + len(edge), "test cố định")

Tất cả cách qua 11 test cố định


### 12.2 Ngẫu nhiên có seed, so sánh với vét cạn

Ba kiểu dữ liệu: nhiều số 0 (dễ có đoạn `P` bằng nhau), giá trị nhỏ `0..5`, và giá trị lớn `0..10⁴`.

In [17]:
rng = random.Random(42)
cnt = 0
for _ in range(500):
    n = rng.randint(3, 25)
    kind = rng.random()
    if kind < 0.3:
        nums = [rng.choice([0, 0, 0, 1, 2]) for _ in range(n)]
    elif kind < 0.7:
        nums = [rng.randint(0, 5) for _ in range(n)]
    else:
        nums = [rng.randint(0, 10**4) for _ in range(n)]
    ref = ways_brute(nums)
    for name, f in impls.items():
        assert f(nums) == ref, (name, nums)
    cnt += 1
for _ in range(60):
    n = rng.randint(100, 400)
    nums = [rng.randint(0, 3) for _ in range(n)]
    ref = ways_prefix_n2(nums)
    for name in ["bisect", "three", "Solution", "SolutionBisect"]:
        assert impls[name](nums) == ref
    cnt += 1
print("OK", cnt, "bộ ngẫu nhiên, mọi cách khớp đáp án chuẩn")

OK 560 bộ ngẫu nhiên, mọi cách khớp đáp án chuẩn


### 12.3 Modulo: mảng `10⁵` số 0

In [18]:
n = 10**5
raw = (n - 1) * (n - 2) // 2
print("C(n-1, 2) =", raw, "| mod =", raw % MOD)
assert Solution().waysToSplit([0] * n) == raw % MOD
assert ways_bisect([0] * n) == raw % MOD
print("Đúng đáp án đã lấy modulo")

C(n-1, 2) = 4999850001 | mod = 999849973


Đúng đáp án đã lấy modulo


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép `right` rỗng (cho `j` chạy tới `n`)

In [19]:
def wrong_right_empty(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums)); T = P[n]
    ans = 0
    for i in range(1, n - 1):
        lo = bisect_left(P, 2 * P[i], i + 1, n + 1)          # SAI: n + 1
        hi = bisect_right(P, (T + P[i]) // 2, lo, n + 1)
        ans += hi - lo
    return ans

for nums in ([0, 0, 0], [0, 0, 0, 0]):
    print(nums, "sai:", wrong_right_empty(nums), "| đúng:", ways_brute(nums))

[0, 0, 0] sai: 2 | đúng: 1
[0, 0, 0, 0] sai: 5 | đúng: 3


**Vì sao sai:** `j = n` nghĩa là `right = nums[n:]` rỗng, trái với yêu cầu "không rỗng". Khi mảng có đủ số `0` (ví dụ toàn `0`), `right` rỗng có tổng `0` vẫn "thỏa" bất đẳng thức nên bị đếm thừa. **Sửa:** tìm `j` trong `[i + 1, n − 1]`, tức tham số `hi` của `bisect` là `n`.

### Lỗi 2 — Dùng `<` thay vì `≤`

In [20]:
def wrong_strict(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums)); T = P[n]
    ans = 0
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            if 2 * P[i] < P[j] and 2 * P[j] < T + P[i]:      # SAI: '<'
                ans += 1
    return ans

print("sai:", wrong_strict([1, 1, 1]), "| đúng:", ways_brute([1, 1, 1]))

sai: 0 | đúng: 1


**Vì sao sai:** đề dùng "less than **or equal**". Ví dụ 1 `[1][1][1]` có ba tổng bằng nhau — chỉ hợp lệ với `≤`. **Sửa:** `2·P[i] ≤ P[j]` và `2·P[j] ≤ T + P[i]` (tương ứng `bisect_left` cho chặn dưới, `bisect_right` cho chặn trên).

### Lỗi 3 — Hai con trỏ không kéo `hi` lên ít nhất bằng `lo`

In [21]:
def wrong_no_clamp(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums)); T = P[n]
    lo = hi = 1
    ans = 0
    for i in range(1, n - 1):
        lo = max(lo, i + 1)
        while lo <= n - 1 and P[lo] < 2 * P[i]:
            lo += 1
        while hi <= n - 1 and 2 * P[hi] <= T + P[i]:   # SAI: thiếu hi = max(hi, lo)
            hi += 1
        ans += hi - lo
    return ans

print("sai:", wrong_no_clamp([2, 1, 0]), "| đúng:", ways_brute([2, 1, 0]))

sai: -1 | đúng: 0


**Vì sao sai:** với `[2,1,0]`, `P = [0,2,3,3]`, `T = 3`. Ở `i = 1`, chặn dưới `P[j] ≥ 4` đẩy `lo` tới `3 = n`, còn chặn trên `2·P[hi] ≤ 5` dừng `hi` ở `2`. `hi − lo = −1` — số cách **âm**! **Sửa:** `hi = max(hi, lo)` trước khi tiến `hi` (hoặc cộng `max(0, hi − lo)`).

### Lỗi 4 — Quên giới hạn `j ≥ i + 1` khi tìm nhị phân

In [22]:
def wrong_no_lo_bound(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums)); T = P[n]
    ans = 0
    for i in range(1, n - 1):
        lo = bisect_left(P, 2 * P[i], 0, n)                  # SAI: bắt đầu từ 0
        hi = bisect_right(P, (T + P[i]) // 2, lo, n)
        ans += hi - lo
    return ans

print("sai:", wrong_no_lo_bound([0, 0, 0, 0]), "| đúng:", ways_brute([0, 0, 0, 0]))

sai: 8 | đúng: 3


**Vì sao sai:** khi có số `0`, `P` có nhiều giá trị bằng nhau ở đầu, nên `bisect_left` từ `0` trả về một `j ≤ i` — tức `mid` rỗng hoặc "âm". **Sửa:** tìm trong `[i + 1, n)`.

### Lỗi 5 — Quên modulo

In [23]:
def wrong_no_mod(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums)); T = P[n]
    ans = 0
    for i in range(1, n - 1):
        if 3 * P[i] > T:
            break
        lo = bisect_left(P, 2 * P[i], i + 1, n)
        hi = bisect_right(P, (T + P[i]) // 2, lo, n)
        ans += hi - lo
    return ans                                           # SAI: thiếu % MOD

z = [0] * 10**5
print("sai:", wrong_no_mod(z), "| đúng:", Solution().waysToSplit(z))

sai: 4999850001 | đúng: 999849973


**Vì sao sai:** Python không tràn số nên con số "trông đúng", nhưng đề yêu cầu trả về modulo `10⁹ + 7`; mảng toàn `0` cho `≈ 5·10⁹` cách. (Trong C++/Java còn bị **tràn `int`** nếu không cộng vào `long`.) **Sửa:** `return ans % MOD`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Chia mảng thành k đoạn liên tiếp"** ⇒ chọn `k − 1` vị trí cắt; tổng đoạn = hiệu tổng tiền tố.
- **Mảng không âm** ⇒ tổng tiền tố không giảm ⇒ điều kiện dạng `L ≤ P[j] ≤ R` cho ra **một đoạn chỉ số** ⇒ `bisect_left`/`bisect_right`.
- **Chặn dưới và chặn trên cùng tăng theo con trỏ ngoài** ⇒ đổi tìm nhị phân thành **hai con trỏ chỉ tiến** ⇒ `O(n)`.
- Viết lại mọi bất đẳng thức sao cho **không chia** (nhân 2 hai vế) hoặc chia nguyên `//` có kiểm chứng.
- "Đếm số cách … modulo" ⇒ đáp án có thể ~`n²`, nhớ `%`.
- Dừng sớm khi điều kiện cần (`3·left ≤ T`) vi phạm.

## 15. Biến thể của bài

1. **Đảo chiều bất đẳng thức** `left ≥ mid ≥ right`: chỉ cần đảo mảng rồi giải bài gốc (bài tập 5).
2. **LC 1013 — Partition Array Into Three Parts With Equal Sum**: ba tổng **bằng nhau** — chặn dưới = chặn trên.
3. **LC 2270 — Number of Ways to Split Array**: chỉ chia thành 2 phần, `left ≥ right`.
4. **LC 548 — Split Array with Equal Sum** (Premium): 4 phần bằng nhau, cần hash set.
5. **Có số âm**: `P` không còn đơn điệu ⇒ không dùng bisect/hai con trỏ được; phải dùng cấu trúc dữ liệu (BIT trên giá trị đã nén) — `O(n log n)`.
6. **Chia thành k đoạn** với tổng không giảm: DP + tổng tiền tố.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn `sum` | thử mọi `(i, j)`, tính tổng trực tiếp | `O(n³)` | `O(n)` | kiểm chứng `n` rất nhỏ |
| 2. Tiền tố `O(n²)` | thử mọi `(i, j)`, tổng `O(1)` | `O(n²)` | `O(n)` | đáp án chuẩn cho test vừa |
| 3. Tiền tố + bisect | đếm đoạn `j` bằng 2 lần tìm nhị phân | `O(n log n)` | `O(n)` | phỏng vấn, dễ chứng minh |
| 4. Ba con trỏ | `lo`, `hi` chỉ tiến | `O(n)` | `O(n)` | **nộp bài**, tối ưu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[0,0,0,0]`; (b) `[2,2,2,2]`; (c) `[10,0,0]`; (d) `[1,1,1,1,1,1]`.

**Bài 2 (tính tay):** Cho `nums = [1,1,1,1,1,1]`. Lập bảng `P`, và với từng `i` ghi chặn dưới, chặn trên, đoạn `j` hợp lệ.

**Bài 3 (điền TODO):** Viết `j_range(P, i, n)` trả về cặp `(lo, hi)` (đoạn nửa mở) các `j` hợp lệ cho `i`, dùng `bisect`.

**Bài 4 (tự viết lại):** Viết lại Cách 4 **không dùng mảng `P`**, chỉ dùng các tổng chạy `s_i`, `s_lo`, `s_hi` (Space `O(1)`).

**Bài 5 (mini challenge):** Đếm số cách chia sao cho `sum(left) ≥ sum(mid) ≥ sum(right)`. Gợi ý: có cách giải chỉ 1 dòng dùng lại `Solution`.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def j_range_todo(P, i, n):
    T = P[n]
    # TODO: lo = bisect_left(...) trong [i+1, n)
    # TODO: hi = bisect_right(...) trong [lo, n)
    return 0, 0

In [25]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def ways_o1_space_todo(nums):
    T = sum(nums)
    # TODO: s_i = tổng left; lo, s_lo; hi, s_hi là các tổng chạy tương ứng
    pass

In [26]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def ways_reverse_order_todo(nums):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: dùng điều kiện `2P[i] ≤ P[j] ≤ (T+P[i])//2`. Mức 2: toàn 0 ⇒ mọi cách chia đều tốt ⇒ `C(n−1, 2)`. Mức 3: (c) `left = 10 > mid = 0` với mọi cách.

**Bài 2.** Mức 1: `P = [0,1,2,3,4,5,6]`, `T = 6`. Mức 2: `i = 1`: `2 ≤ P[j] ≤ 3`. Mức 3: `i = 2`: `4 ≤ P[j] ≤ 4`; `i = 3`: `3·3 = 9 > 6` dừng.

**Bài 3.** Mức 1: chặn dưới `2·P[i]`. Mức 2: chặn trên `(T + P[i]) // 2`. Mức 3: tham số `hi = n` của bisect để `right` không rỗng.

**Bài 4.** Mức 1: `s_lo` là `P[lo]`, cập nhật `s_lo += nums[lo]` khi `lo += 1` (vì `P[lo+1] = P[lo] + nums[lo]`). Mức 2: khi `lo = max(lo, i + 1)` nhảy, phải cộng bù vào `s_lo`. Mức 3: đối chiếu với `Solution` trên nhiều mảng ngẫu nhiên.

**Bài 5.** Mức 1: đảo mảng thì `left` ↔ `right`. Mức 2: `L' ≤ M' ≤ R'` trên mảng đảo ⇔ `R ≤ M ≤ L` trên mảng gốc. Mức 3: `Solution().waysToSplit(nums[::-1])`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** (a) `3`; (b) `1` (`[2][2][2,2]`); (c) `0`; (d) `[1][1][1,1,1,1]`, `[1][1,1][1,1,1]`, `[1,1][1,1][1,1]` ⇒ `3`.

In [27]:
assert Solution().waysToSplit([0, 0, 0, 0]) == 3
assert Solution().waysToSplit([2, 2, 2, 2]) == 1
assert Solution().waysToSplit([10, 0, 0]) == 0
assert Solution().waysToSplit([1] * 6) == 3
print("Bài 1 đúng")

Bài 1 đúng


**Bài 2:** `P = [0,1,2,3,4,5,6]`, `T = 6`.

| `i` | chặn dưới | chặn trên | `j` hợp lệ |
|---|---|---|---|
| 1 | 2 | 3 | `j = 2, 3` |
| 2 | 4 | 4 | `j = 4` |
| 3 | dừng (`9 > 6`) | | |

Tổng 3 — khớp Bài 1(d).

In [28]:
print(trace_three_pointers([1] * 6))

P = [0, 1, 2, 3, 4, 5, 6] T = 6
i=1: P[i]= 1 cần 2 <= P[j] <= 3  lo=2 hi=4 j=[2, 3] +2 ans=2
       [1] [1] [1, 1, 1, 1]
       [1] [1, 1] [1, 1, 1]
i=2: P[i]= 2 cần 4 <= P[j] <= 4  lo=4 hi=5 j=[4] +1 ans=3
       [1, 1] [1, 1] [1, 1]
i=3: 3*P[i]=9 > T=6 -> dừng
3


In [29]:
# Bài 3 — đáp án
def j_range(P, i, n):
    T = P[n]
    lo = bisect_left(P, 2 * P[i], i + 1, n)
    hi = bisect_right(P, (T + P[i]) // 2, lo, n)
    return lo, hi

P = [0] + list(accumulate([1, 2, 2, 2, 5, 0]))
print([j_range(P, i, 6) for i in range(1, 5)])
assert [j_range(P, i, 6) for i in range(1, 4)] == [(2, 4), (4, 5), (5, 5)]
print("Bài 3 đúng")

[(2, 4), (4, 5), (5, 5), (6, 6)]
Bài 3 đúng


In [30]:
# Bài 4 — đáp án: O(1) bộ nhớ phụ
def ways_o1_space(nums):
    n = len(nums)
    T = sum(nums)
    ans = 0
    lo = hi = 0
    s_lo = s_hi = 0          # s_lo = P[lo], s_hi = P[hi]
    s_i = 0                  # s_i = P[i]
    for i in range(1, n - 1):
        s_i += nums[i - 1]
        if 3 * s_i > T:
            break
        while lo < i + 1:
            s_lo += nums[lo]; lo += 1
        while lo <= n - 1 and s_lo < 2 * s_i:
            s_lo += nums[lo]; lo += 1
        if hi < lo:
            hi, s_hi = lo, s_lo
        while hi <= n - 1 and 2 * s_hi <= T + s_i:
            s_hi += nums[hi]; hi += 1
        ans += hi - lo
    return ans % MOD

rng = random.Random(4)
for _ in range(400):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(3, 30))]
    assert ways_o1_space(nums) == Solution().waysToSplit(nums)
for nums, want in examples:
    assert ways_o1_space(nums) == want
print("Bài 4 đúng")

Bài 4 đúng


In [31]:
# Bài 5 — đáp án
def ways_reverse_order(nums):
    return Solution().waysToSplit(nums[::-1])

def brute_reverse_order(nums):
    n = len(nums)
    return sum(1 for i in range(1, n - 1) for j in range(i + 1, n)
               if sum(nums[:i]) >= sum(nums[i:j]) >= sum(nums[j:]))

rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(0, 6) for _ in range(rng.randint(3, 15))]
    assert ways_reverse_order(nums) == brute_reverse_order(nums)
print("[3,2,1] ->", ways_reverse_order([3, 2, 1]))
print("Bài 5 đúng")

[3,2,1] -> 1
Bài 5 đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# Đếm cách chia 3 đoạn left <= mid <= right (mảng không âm)
P = [0] + list(accumulate(nums)); T = P[n]
lo = hi = 0; ans = 0
for i in range(1, n - 1):                    # i = độ dài left
    if 3 * P[i] > T: break
    lo = max(lo, i + 1)
    while lo <= n - 1 and P[lo] < 2 * P[i]: lo += 1          # left <= mid
    hi = max(hi, lo)
    while hi <= n - 1 and 2 * P[hi] <= T + P[i]: hi += 1     # mid <= right
    ans += hi - lo
return ans % MOD

# Bản bisect
lo = bisect_left(P, 2 * P[i], i + 1, n)
hi = bisect_right(P, (T + P[i]) // 2, lo, n)
```

**Checklist trước khi submit:**
- [ ] `i ∈ [1, n−2]`, `j ∈ [i+1, n−1]` — cả ba đoạn **không rỗng**.
- [ ] Dấu `≤` cho cả hai điều kiện (`bisect_left` chặn dưới, `bisect_right` chặn trên).
- [ ] Chặn trên dùng `(T + P[i]) // 2` (không phải `(T − P[i]) // 2`).
- [ ] Hai con trỏ: `hi = max(hi, lo)`, không bao giờ cộng số âm.
- [ ] `% (10⁹ + 7)` ở cuối.
- [ ] Thử `[0,0,0]`, `[0,0,0,0]`, `[2,1,0]`, và 3 ví dụ của đề.